# 03 — Frozen controlled comparator benchmark

Question: does label-free quality weighting improve held-out predictions when the quality signal is informative, irrelevant, or misleading?

Inputs: deterministic compact mathematical arrays, NOT the signal-derived EEG/fMRI features in 01/02. No protected data or E01 results. Outputs: aggregate 03 JSON evidence, all 12 metrics, paired intervals, calibration/confusion/gate diagnostics and editable figures. See [literature map](../docs/S01_LITERATURE_EVIDENCE_PLAN.md). No package from that map is claimed as an executed comparator.

In [1]:
from pathlib import Path
import sys
import json
from IPython.display import Markdown, display
START = Path.cwd().resolve()
ROOT = next(p for p in (START, *START.parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'scripts'))
import s01_comparator_benchmark as benchmark
protocol = benchmark.load_protocol()
{'protocol_commit': benchmark.PROTOCOL_COMMIT, 'protocol_sha256': benchmark.PROTOCOL_HASH, 'settings': protocol}

{'protocol_commit': 'c4d0ca6',
 'protocol_sha256': 'c492d518958916df3998c5dac11cb0e01c4734481de9669e429ed3d3ebb0849b',
 'settings': {'schema_version': '1.0',
  'protocol_id': 's01-synthetic-comparator-v1',
  'question': 'How do equal, label-free quality-aware, and training-tuned constant-weight fusion behave under informative, unrelated, and misleading quality in controlled mathematical classification fixtures?',
  'scope': 'synthetic_compact_interface_arrays_not_eeg_fmri_population_validation',
  'data_license': 'repository BSD-3-Clause; generated mathematical fixtures, no participant data',
  'fixture_seeds': [20261011, 20261012, 20261013],
  'records_per_fixture': 120,
  'positive_fraction': 0.5,
  'feature_dimensions': {'eeg': 6, 'fmri': 8},
  'signal_coefficients': {'eeg': [0.9, 0.5, 0, 0, 0, 0],
   'fmri': [0.9, 0.5, 0.3, 0, 0, 0, 0, 0]},
  'noise_sigma': '0.4 + 2.4 * degradation; degradation independently Uniform(0,1) per record and modality',
  'quality_conditions': ['informati

## Matched, training-only comparison

Three fixture seeds × 120 balanced records; two repetitions of four outer folds, three inner folds; all five comparators share unimodal predictions and eligible records. C, constant fusion weight and decision thresholds are selected on training OOF predictions. Quality scaling is fitted inside training folds without labels. Equal and negative outcomes remain in the report. Protocol was committed before first execution.

In [2]:
results = benchmark.run_benchmark()
result_path = benchmark.write_results(results)
assert len(results['folds']) == 72
assert all(row['records'] == 360 for row in results['conditions'].values())
assert all(len(row['profiles']) == 5 for row in results['conditions'].values())
{'artifact': result_path.relative_to(ROOT).as_posix(), 'records_per_condition': 360, 'outer_fold_runs': 72, 'metrics': protocol['metrics'], 'wall_seconds': results['metadata']['wall_seconds_including_uncertainty']}

{'artifact': 'notebooks/artifacts/03_comparator_results.json',
 'records_per_condition': 360,
 'outer_fold_runs': 72,
 'metrics': ['roc_auc',
  'average_precision',
  'brier_score',
  'log_loss',
  'accuracy',
  'balanced_accuracy',
  'precision',
  'sensitivity',
  'specificity',
  'f1',
  'mcc',
  'ece_10_uniform'],
 'wall_seconds': 125.19254650000948}

Benchmark condition: informative
Benchmark condition: unrelated
Benchmark condition: misleading
Saved notebooks/artifacts/03_comparator_results.json


## Predeclared primary result and uncertainty boundary

Brier is the primary score. Report paired candidate-minus-equal differences, including misleading quality. The 1,000 paired percentile bootstrap resamples preserve seed/class strata and resample each record once after averaging repeated predictions. Intervals are conditional on fitted models: not refitting uncertainty, independent-fold inference, population or clinical generalization. No multiplicity-adjusted significance claim. Log loss clips probabilities to [1e−6,1−1e−6]; ECE uses ten equal-width bins. Threshold metrics use repeated-fold majority decisions, ties positive; precision/F1/MCC are zero when undefined.

In [3]:
for condition, row in results['conditions'].items():
    difference = next(d for d in row['paired_differences'] if d['candidate'] == 'quality' and d['metric'] == 'brier_score')
    print(condition, json.dumps(difference, sort_keys=True))
print('Gate diagnostics use held-out unimodal utility, independent of the gate.')
{condition: row['gate_validity'] for condition, row in results['conditions'].items()}

{'informative': {'positive_utility_records': 214,
  'records': 360,
  'utility_auc': {'value': 0.5534502624503904,
   'ci_low': 0.4945423350589629,
   'ci_high': 0.6152769961933736,
   'valid_resamples': 1000},
  'spearman_rho': {'value': 0.14176138190366694,
   'ci_low': 0.04237350049189776,
   'ci_high': 0.24628472370298915,
   'valid_resamples': 1000}},
 'unrelated': {'positive_utility_records': 214,
  'records': 360,
  'utility_auc': {'value': 0.5241646396108053,
   'ci_low': 0.46951779832324186,
   'ci_high': 0.5874018605206928,
   'valid_resamples': 1000},
  'spearman_rho': {'value': 0.005088259425870055,
   'ci_low': -0.09495428142701766,
   'ci_high': 0.10607364560088947,
   'valid_resamples': 1000}},
 'misleading': {'positive_utility_records': 214,
  'records': 360,
  'utility_auc': {'value': 0.44728587888874666,
   'ci_low': 0.3864458737580024,
   'ci_high': 0.5055015202989895,
   'valid_resamples': 1000},
  'spearman_rho': {'value': -0.1459628032109301,
   'ci_low': -0.24993

informative {"candidate": "quality", "ci_high": -0.0027050800577367654, "ci_low": -0.0066719173040774874, "difference": -0.004477479796043793, "interval_spans_zero": false, "metric": "brier_score", "reference": "equal"}
unrelated {"candidate": "quality", "ci_high": 0.002109422639055274, "ci_low": -0.0014795190311809362, "difference": 0.00032271591772853325, "interval_spans_zero": true, "metric": "brier_score", "reference": "equal"}
misleading {"candidate": "quality", "ci_high": 0.007915794684698916, "ci_low": 0.0037378661649286027, "difference": 0.005577608550758867, "interval_spans_zero": false, "metric": "brier_score", "reference": "equal"}
Gate diagnostics use held-out unimodal utility, independent of the gate.


## Full evidence figure gallery

All twelve metrics and all five methods are graphed, followed by ROC/PR, calibration with bin support, confusion counts, utility validity and training-only tuning traces. [Full captioned gallery](figures/README.md). PDF/SVG and exact evidence hashes accompany each PNG. Mathematical quality conditions are ablations, not cross-dataset validation.

In [4]:
from s01_evidence_figures import generate
figure_records = generate('03')
for name in ('F07_paired_brier', 'F09_calibration_support', 'F11_gate_validity'):
    item = next(r for r in figure_records if r['id'] == name)
    display(Markdown(f"### {name}\n\n![{name}](figures/previews/{name}.png)\n\n{item['caption']}"))
{'figure_sets': len(figure_records), 'gallery': 'notebooks/figures/README.md'}

### F07_paired_brier

![F07_paired_brier](figures/previews/F07_paired_brier.png)

Paired candidate-minus-equal Brier differences with 95% record-bootstrap intervals. Negative is lower error. Same resampled indices for every method; intervals are descriptive, no multiplicity-adjusted significance claim. Misleading-quality failures remain visible.

### F09_calibration_support

![F09_calibration_support](figures/previews/F09_calibration_support.png)

Ten equal-width reliability bins paired with their actual support. Empty bins are omitted from calibration curves but retained as zero in count panels; probability 1 belongs to the last bin. These curves are descriptive, with no bin-wise uncertainty claim; ECE depends on binning.

### F11_gate_validity

![F11_gate_validity](figures/previews/F11_gate_validity.png)

Utility is |y−p_EEG|−|y−p_fMRI|, computed from held-out unimodal predictions independently of the gate. Upper row: descriptive means in weight bins, no CI. Lower: utility-sign AUC (chance .5) and Spearman association (null 0), paired-record bootstrap intervals. Quality monotonicity alone does not validate utility.

{'figure_sets': 9, 'gallery': 'notebooks/figures/README.md'}

Exported F06_metrics_informative: Times New Roman, 1000 dpi
Exported F06_metrics_unrelated: Times New Roman, 1000 dpi
Exported F06_metrics_misleading: Times New Roman, 1000 dpi
Exported F07_paired_brier: Times New Roman, 1000 dpi
Exported F08_discrimination_curves: Times New Roman, 1000 dpi
Exported F09_calibration_support: Times New Roman, 1000 dpi
Exported F10_confusion_counts: Times New Roman, 1000 dpi
Exported F11_gate_validity: Times New Roman, 1000 dpi
Exported F12_nested_tuning: Times New Roman, 1000 dpi


## Next gate

Notebook 04 will add predeclared robustness, resource measurements and release evidence; it is not complete yet. A sixth notebook (05) is reserved for real cross-dataset validation only when two licensed compatible paired-modality datasets and an untouched-target protocol are available. No synthetic scenario is renamed as a dataset.